# ARX R5 Dual-Arm Test — ee_pose 模式

Server 端需以 `--control-mode normal --arms-only` 启动, 开启 ee_pose 支持.

In [1]:
import importlib.util, sys, os

# Direct file import — avoids robots/__init__.py (which pulls in zerorpc/placo)
_client_path = os.path.abspath(os.path.join(os.getcwd(), "arx_interface_client.py"))
_spec = importlib.util.spec_from_file_location("arx_interface_client", _client_path)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
ArxDualArmClient = _mod.ArxDualArmClient

import numpy as np
import time

ROBOT_IP = "192.168.110.57"
ROBOT_PORT = 4242

## 1. Connect & Read State

In [2]:
client = ArxDualArmClient(ROBOT_IP, ROBOT_PORT)
ok = client.system_connect(timeout=10.0)
print(f"Connected: {ok}")

state = client.get_full_state()
for side in ('left_arm', 'right_arm'):
    arm = state[side]
    print(f"\n=== {side} ===")
    print(f"  joint_pos (7): {np.round(arm['joint_positions'], 4)}")
    print(f"  end_pose  (6): {np.round(arm['end_pose'], 4)}  [x,y,z,roll,pitch,yaw]")
    print(f"  gripper:       {arm['gripper']:.4f}")

Connected: True

=== left_arm ===
  joint_pos (7): [-0.0021 -0.001   0.0147 -0.0048 -0.0006  0.0063 -0.0011]
  end_pose  (6): [-0.001  -0.0002  0.0046  0.0063 -0.0109 -0.0015]  [x,y,z,roll,pitch,yaw]
  gripper:       -0.0002

=== right_arm ===
  joint_pos (7): [-0.0021  0.0002  0.0082 -0.0048 -0.0013 -0.0135 -0.0008]
  end_pose  (6): [-0.0005 -0.0002  0.0023 -0.0135 -0.0032 -0.0008]  [x,y,z,roll,pitch,yaw]
  gripper:       -0.0002


## 2. Gripper Control

0.0 = open, 1.0 = closed

In [3]:
for side, set_fn, get_fn in [
    ("left",  client.set_left_gripper,  client.get_left_gripper_position),
    ("right", client.set_right_gripper, client.get_right_gripper_position),
]:
    print(f"--- {side} gripper ---")
    set_fn(1.0); time.sleep(0.8)
    print(f"  closed: {get_fn():.4f}")
    set_fn(0.0); time.sleep(0.8)
    print(f"  opened: {get_fn():.4f}")

--- left gripper ---
  closed: 0.9803
  opened: -0.0007
--- right gripper ---
  closed: 0.9802
  opened: -0.0005


## 3. ee_pose 读取验证

确认 `get_full_state().end_pose` 和 `get_left/right_end_pose()` 一致

In [4]:
state = client.get_full_state()
left_ee = np.array(state["left_arm"]["end_pose"])
right_ee = np.array(state["right_arm"]["end_pose"])
left_ee_api = client.get_left_end_pose()
right_ee_api = client.get_right_end_pose()

print("Left  ee_pose (full_state):", np.round(left_ee, 4))
print("Left  ee_pose (API):       ", np.round(left_ee_api, 4))
print("  match:", np.allclose(left_ee, left_ee_api, atol=0.01))

print("\nRight ee_pose (full_state):", np.round(right_ee, 4))
print("Right ee_pose (API):       ", np.round(right_ee_api, 4))
print("  match:", np.allclose(right_ee, right_ee_api, atol=0.01))

Left  ee_pose (full_state): [-0.0009 -0.0002  0.0036  0.0067 -0.0002 -0.0015]
Left  ee_pose (API):        [-0.0009 -0.0002  0.0036  0.0067 -0.0002 -0.0015]
  match: True

Right ee_pose (full_state): [-0.0002 -0.0001  0.0011 -0.015   0.0105  0.0004]
Right ee_pose (API):        [-0.0002 -0.0001  0.0011 -0.015   0.0105  0.0004]
  match: True


## 4. ee_pose 写入 — 发送当前位姿 (机械臂不动)

验证 `set_dual_ee_poses` RPC 通路是否正常: 把当前读到的位姿原样发回去, 机械臂应该纹丝不动.

In [5]:
state = client.get_full_state()
left_ee = state["left_arm"]["end_pose"]     # list [x,y,z,r,p,y]
right_ee = state["right_arm"]["end_pose"]
left_grip = state["left_arm"]["gripper"]
right_grip = state["right_arm"]["gripper"]

print("Sending current ee_pose back (robot should NOT move)...")
client.set_dual_ee_poses(left_ee, right_ee, left_grip, right_grip)
time.sleep(0.5)

# 读回验证
state2 = client.get_full_state()
print(f"Left  ee before: {np.round(left_ee, 4)}")
print(f"Left  ee after:  {np.round(state2['left_arm']['end_pose'], 4)}")
print(f"Right ee before: {np.round(right_ee, 4)}")
print(f"Right ee after:  {np.round(state2['right_arm']['end_pose'], 4)}")
print("OK — no error means RPC path works.")

Sending current ee_pose back (robot should NOT move)...
Left  ee before: [-0.0009 -0.0002  0.0036  0.0067 -0.0002 -0.0015]
Left  ee after:  [-0.0003 -0.0002  0.0039  0.0067 -0.0013 -0.0015]
Right ee before: [-0.0002 -0.0001  0.0011 -0.015   0.0101  0.0004]
Right ee after:  [-0.0002 -0.0001  0.0011 -0.015   0.0105  0.0004]
OK — no error means RPC path works.


## 5. ee_pose 小幅运动 — 右臂 x 方向前进 2cm 再返回

**机械臂会动!** 确保周围安全.

In [7]:
state = client.get_full_state()
left_ee = np.array(state["left_arm"]["end_pose"])
right_ee_origin = np.array(state["right_arm"]["end_pose"])
left_grip = state["left_arm"]["gripper"]
right_grip = state["right_arm"]["gripper"]

# 右臂 x +2cm
right_ee_target = right_ee_origin.copy()
right_ee_target[0] += 0.02

print(f"Right arm x: {right_ee_origin[0]:.4f} -> {right_ee_target[0]:.4f} (+2cm)")
print("Moving...")
client.set_dual_ee_poses(left_ee.tolist(), right_ee_target.tolist(), left_grip, right_grip)
time.sleep(1.0)

state_after = client.get_full_state()
print(f"Right arm x after: {state_after['right_arm']['end_pose'][0]:.4f}")

# 移回原位
print("Moving back...")
client.set_dual_ee_poses(left_ee.tolist(), right_ee_origin.tolist(), left_grip, right_grip)
time.sleep(1.0)

state_back = client.get_full_state()
print(f"Right arm x back:  {state_back['right_arm']['end_pose'][0]:.4f}")
print(f"Delta from origin: {abs(state_back['right_arm']['end_pose'][0] - right_ee_origin[0]):.4f} m")

Right arm x: 0.0000 -> 0.0200 (+2cm)
Moving...
Right arm x after: 0.0201
Moving back...
Right arm x back:  -0.0000
Delta from origin: 0.0000 m


## 6. ee_pose 连续发送 — 模拟遥操 30Hz

以 30Hz 连续发送小增量 delta, 测试 ee_pose 模式下的流畅度和延迟.
右臂 x 方向: 前进 3cm -> 返回, 左臂不动.

In [8]:
FREQ = 30          # Hz
DURATION = 3.0      # 秒
AMPLITUDE = 0.03    # 米 (前进3cm再返回)

state = client.get_full_state()
left_ee = state["left_arm"]["end_pose"]
right_ee_base = np.array(state["right_arm"]["end_pose"])
left_grip = state["left_arm"]["gripper"]
right_grip = state["right_arm"]["gripper"]

steps = int(FREQ * DURATION)
dt = 1.0 / FREQ
speed = AMPLITUDE / (DURATION / 2)   # m/s

timings = []
t_start = time.perf_counter()

print(f"Running {steps} steps at {FREQ}Hz, amplitude={AMPLITUDE*100:.0f}cm ...")

for step in range(steps):
    t0 = time.perf_counter()
    elapsed = t0 - t_start

    # 三角波轨迹
    if elapsed < DURATION / 2:
        dx = speed * elapsed
    else:
        dx = speed * (DURATION - elapsed)

    right_ee_target = right_ee_base.copy()
    right_ee_target[0] += dx

    client.set_dual_ee_poses(left_ee, right_ee_target.tolist(), left_grip, right_grip)

    rpc_ms = (time.perf_counter() - t0) * 1e3
    timings.append(rpc_ms)

    # 等到下一步
    remaining = dt - (time.perf_counter() - t0)
    if remaining > 0:
        time.sleep(remaining)

# 复位
client.set_dual_ee_poses(left_ee, right_ee_base.tolist(), left_grip, right_grip)

t_arr = np.array(timings)
print(f"\nset_dual_ee_poses latency ({steps} calls):")
print(f"  avg: {t_arr.mean():.2f} ms")
print(f"  p50: {np.median(t_arr):.2f} ms")
print(f"  p95: {np.percentile(t_arr, 95):.2f} ms")
print(f"  max: {t_arr.max():.2f} ms")
overruns = np.sum(t_arr > (1000 / FREQ))
print(f"  overruns (>{1000/FREQ:.0f}ms): {overruns}/{steps}")

Running 90 steps at 30Hz, amplitude=3cm ...

set_dual_ee_poses latency (90 calls):
  avg: 0.51 ms
  p50: 0.46 ms
  p95: 0.79 ms
  max: 0.87 ms
  overruns (>33ms): 0/90


## 7. ee_pose + 夹爪联合控制

验证 `set_dual_ee_poses` 的 gripper 参数能同时控制夹爪.
右臂前进 2cm 同时关闭夹爪, 然后复位并打开夹爪.

In [9]:
state = client.get_full_state()
left_ee = state["left_arm"]["end_pose"]
right_ee = np.array(state["right_arm"]["end_pose"])

# 右臂前进 + 关闭双夹爪
right_target = right_ee.copy()
right_target[0] += 0.02

print("Moving right arm +2cm x AND closing both grippers...")
client.set_dual_ee_poses(left_ee, right_target.tolist(), 1.0, 1.0)
time.sleep(1.5)

s = client.get_full_state()
print(f"  Right x: {s['right_arm']['end_pose'][0]:.4f}")
print(f"  Left gripper:  {s['left_arm']['gripper']:.4f}")
print(f"  Right gripper: {s['right_arm']['gripper']:.4f}")

# 复位 + 打开夹爪
print("Restoring position AND opening grippers...")
client.set_dual_ee_poses(left_ee, right_ee.tolist(), 0.0, 0.0)
time.sleep(1.5)

s = client.get_full_state()
print(f"  Right x: {s['right_arm']['end_pose'][0]:.4f}")
print(f"  Left gripper:  {s['left_arm']['gripper']:.4f}")
print(f"  Right gripper: {s['right_arm']['gripper']:.4f}")

Moving right arm +2cm x AND closing both grippers...
  Right x: 0.0204
  Left gripper:  0.9803
  Right gripper: 0.9802
Restoring position AND opening grippers...
  Right x: 0.0001
  Left gripper:  -0.0007
  Right gripper: -0.0006


## 8. Latency Test — 读 vs 写

In [10]:
N = 100
state = client.get_full_state()
left_ee = state["left_arm"]["end_pose"]
right_ee = state["right_arm"]["end_pose"]

# Read latency
read_lat = []
for _ in range(N):
    t0 = time.perf_counter()
    client.get_full_state()
    read_lat.append((time.perf_counter() - t0) * 1e3)

# Write latency (fire-and-forget, 发当前位姿不动)
write_lat = []
for _ in range(N):
    t0 = time.perf_counter()
    client.set_dual_ee_poses(left_ee, right_ee, 0.0, 0.0)
    write_lat.append((time.perf_counter() - t0) * 1e3)

for name, lat in [("get_full_state (read)", read_lat), ("set_dual_ee_poses (write)", write_lat)]:
    a = np.array(lat)
    print(f"{name} x{N}:")
    print(f"  avg={a.mean():.2f}ms  p50={np.median(a):.2f}ms  p95={np.percentile(a,95):.2f}ms  max={a.max():.2f}ms")
    print(f"  achievable Hz: {1000/a.mean():.0f}")

get_full_state (read) x100:
  avg=0.30ms  p50=0.28ms  p95=0.55ms  max=0.75ms
  achievable Hz: 3305
set_dual_ee_poses (write) x100:
  avg=0.27ms  p50=0.21ms  p95=0.53ms  max=0.76ms
  achievable Hz: 3637


## 9. Disconnect

In [ ]:
client.disconnect()
client.close()
print("Disconnected.")